# Portfolio Assessment 3 - Advanced ML & Deep Learning
## LSTM for Regression (Task 3b)

**Cover page**
| | |
|---|---|
| Unit | COS40007 Artificial Intelligence Engineering / COS40004 Topics in Computer Science |
| Task | Portfolio Assessment 3 (Task 3b) |
| Studio / Class | 1 (Monday 17:30) |
| Student | Binh Thai Nguyen |
| Student ID | 104188845 |
| Lecturer | Dr Afzal Azeem Chowdhary |
| Submission date | 5 October 2026 |
| Dataset | Power Consumption of Tetouan City (UCI #849) |
| Problem type | Regression on sequential (time-series) data |

### Acknowledgement of Country
I acknowledge the Wurundjeri Woi Wurrung people of the Kulin Nation, the Traditional Custodians of the land on which I live, learn, and work. I pay my respects to Elders past, present, and emerging.

### Why Task 3b
The dataset is **sensor time-series**: 52,416 observations recorded every 10 minutes across 2017. The target `TotalPower` is **continuous** (watts). The selection table in the brief maps *Sequential/Time-Series + regression target* to **Task 3b - LSTM for Regression**, so that is the branch followed here.

### Continuity with earlier portfolio work
| Portfolio | What it produced | Used here |
|---|---|---|
| 1 - EDA | Cleaned `tetouan_power.csv` (52,416 x 10, 0 missing, 0 duplicates); target `TotalPower`; top correlations Hour +0.68, Temperature +0.49, Humidity -0.30 | Input dataset |
| 2 - Preprocessing and modelling | Engineered features `CoolingDegree`, `TempHour`, `SolarTotal`, `DiffuseRatio`, `HourSin`, `HourCos`; Decision Tree / SVM on a 4-class version | Feature set reused, now for regression |

Portfolio 1 explicitly listed **lagged load (t-1h, t-24h)** and a **time-based split** as missing work. Portfolio 3 acts on both.


## 0. Environment, reproducibility and paths

In [ ]:
# Core imports. Warnings are silenced only for sklearn/TF deprecation noise,
# not for anything that would hide a modelling error.
import os, json, time, pickle, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow import keras

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

# Output folders: models/ for .pkl + .h5, figures/ for report screenshots,
# results/ for the CSV tables quoted in the PDF.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'tetouan_power.csv').exists() and (ROOT.parent / 'data' / 'tetouan_power.csv').exists():
    ROOT = ROOT.parent
MODELS, FIGS, RESULTS = ROOT / 'models', ROOT / 'figures', ROOT / 'results'
for d in (MODELS, FIGS, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('Project root:', ROOT)

---
# Part A - Comparing Traditional Machine Learning Models

## A.1 Load and analyse the dataset

`tetouan_power.csv` is the cleaned file produced in Portfolio Assessment 1 (raw UCI
columns renamed, the three zone meters summed into `TotalPower`, and `Hour`,
`DayOfWeek`, `Month`, `IsWeekend` derived from `DateTime`).

Portfolio 1 dropped the `DateTime` column after extracting the calendar features. The
rows are still in strict chronological order on a perfectly regular 10-minute grid, so
the timestamp index is rebuilt here - an LSTM needs the ordering to be explicit.

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'tetouan_power.csv')

# Rebuild the timestamp: the UCI series starts 2017-01-01 00:00 and is sampled
# every 10 minutes with no gaps (verified against the raw file in Portfolio 1).
df['DateTime'] = pd.date_range('2017-01-01 00:00', periods=len(df), freq='10min')
df = df.set_index('DateTime')

print(f'Rows: {len(df):,}   Columns: {df.shape[1]}')
print(f'Period: {df.index[0]}  ->  {df.index[-1]}')
print(f'Sampling interval: {df.index.to_series().diff().mode()[0]}')
print(f'Missing values: {int(df.isna().sum().sum())}   Duplicated timestamps: {int(df.index.duplicated().sum())}')
df.head()

In [ ]:
TARGET = 'TotalPower'

print('Target variable :', TARGET)
print('dtype           :', df[TARGET].dtype)
print('Unique values   :', df[TARGET].nunique(), '-> continuous, not categorical')
print('Problem type    : REGRESSION (sequential / time-series)')
print()
df[TARGET].describe().to_frame().T

In [ ]:
# Visual confirmation that the target is a continuous, strongly seasonal signal.
fig, ax = plt.subplots(2, 1, figsize=(13, 6.5))

df[TARGET].plot(ax=ax[0], lw=0.3, color='#1f77b4')
ax[0].set(title='TotalPower across 2017 (10-minute resolution)', ylabel='Watts')

week = df.loc['2017-07-03':'2017-07-10', TARGET]
week.plot(ax=ax[1], lw=1.0, color='#d62728')
ax[1].set(title='One summer week - clear daily cycle the LSTM must learn',
          ylabel='Watts', xlabel='')

plt.tight_layout()
plt.savefig(FIGS / 'fig_01_target_timeseries.png', bbox_inches='tight')
plt.show()

## A.2 Pre-process the data

**Feature engineering** is carried over unchanged from Portfolio 2 so the three
assessments describe one consistent system:

| Feature | Formula | Engineering reason |
|---|---|---|
| `CoolingDegree` | `max(T - 18, 0)` | air-conditioning only switches on above a comfort threshold |
| `TempHour` | `T x Hour` | afternoon heat coinciding with occupancy (highest correlation in P2: r = 0.746) |
| `SolarTotal` | `GDF + DF` | total irradiance |
| `DiffuseRatio` | `DF / (GDF + 1)` | cloudy vs clear-sky mix |
| `HourSin`, `HourCos` | cyclic encoding of `Hour` | hour 23 and hour 0 are adjacent, not 23 units apart |

**Split strategy.** A random `train_test_split` would place future observations in the
training set and leak information backwards - fatal for a forecasting model. A
**chronological 80 / 20 split** is used instead: the model trains on Jan-Oct and is
tested on Nov-Dec, which is how the system would actually be deployed. The same split
index is reused for every model in this notebook, so all results are directly comparable.

**Scaling.** `StandardScaler` is fitted on the **training rows only** and then applied to
the test rows, so no test statistics leak into training.

In [ ]:
# --- engineered features (identical definitions to Portfolio Assessment 2) ---
df['CoolingDegree'] = np.maximum(df['Temperature'] - 18.0, 0)
df['TempHour']      = df['Temperature'] * df['Hour']
df['SolarTotal']    = df['GeneralDiffuseFlows'] + df['DiffuseFlows']
df['DiffuseRatio']  = df['DiffuseFlows'] / (df['GeneralDiffuseFlows'] + 1)
df['HourSin']       = np.sin(2 * np.pi * df['Hour'] / 24)
df['HourCos']       = np.cos(2 * np.pi * df['Hour'] / 24)

ORIGINAL   = ['Temperature','Humidity','WindSpeed','GeneralDiffuseFlows','DiffuseFlows',
              'Hour','DayOfWeek','Month','IsWeekend']
ENGINEERED = ['CoolingDegree','TempHour','SolarTotal','DiffuseRatio','HourSin','HourCos']
FEATURES   = ORIGINAL + ENGINEERED

print(f'{len(ORIGINAL)} original + {len(ENGINEERED)} engineered = {len(FEATURES)} predictors')
df[ENGINEERED].describe().T

In [ ]:
# --- chronological split shared by every model in this notebook ---
N         = len(df)
SPLIT_IDX = int(N * 0.80)

X = df[FEATURES].to_numpy(dtype='float64')
y = df[TARGET].to_numpy(dtype='float64')

X_train_raw, X_test_raw = X[:SPLIT_IDX], X[SPLIT_IDX:]
y_train,     y_test     = y[:SPLIT_IDX], y[SPLIT_IDX:]

# Scaler fitted on training rows only -> no leakage from the test period.
scaler_std   = StandardScaler().fit(X_train_raw)
X_train      = scaler_std.transform(X_train_raw)
X_test       = scaler_std.transform(X_test_raw)

print(f'Training set : {X_train.shape[0]:,} samples x {X_train.shape[1]} features'
      f'   ({df.index[0].date()} -> {df.index[SPLIT_IDX-1].date()})')
print(f'Test set     : {X_test.shape[0]:,} samples x {X_test.shape[1]} features'
      f'   ({df.index[SPLIT_IDX].date()} -> {df.index[-1].date()})')
print(f'Split ratio  : {X_train.shape[0]/N:.0%} / {X_test.shape[0]/N:.0%} (chronological, not shuffled)')
print(f'\ny_train mean {y_train.mean():,.0f} W | y_test mean {y_test.mean():,.0f} W')

## A.3 Train two traditional ML models

The brief specifies **Random Forest Regressor** and **MLP Regressor** for regression
problems. Both are trained on the scaled chronological training set. The MLP needs the
scaling; the Random Forest does not, but using the same matrix keeps the comparison
clean.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=None, min_samples_leaf=2,
    random_state=RANDOM_STATE, n_jobs=-1,
)
t0 = time.time(); rf.fit(X_train, y_train)
print(f'Random Forest trained in {time.time()-t0:.1f}s')

mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
    alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=300,
    early_stopping=True, n_iter_no_change=15, random_state=RANDOM_STATE,
)
t0 = time.time(); mlp.fit(X_train, y_train)
print(f'MLP Regressor trained in {time.time()-t0:.1f}s  ({mlp.n_iter_} iterations)')

pred_rf  = rf.predict(X_test)
pred_mlp = mlp.predict(X_test)

## A.4 Evaluate and compare

Metrics required by the brief for regression: **R^2**, **RMSE**, **MAE** and a
**5-fold cross-validation** score.

Ordinary `KFold` would again shuffle time. `TimeSeriesSplit(n_splits=5)` is the
time-series form of 5-fold CV: each fold trains on a growing prefix and validates on the
block that immediately follows it, so every validation fold stays in the future of its
training data.

In [ ]:
def regression_metrics(name, y_true, y_pred):
    """R2 / RMSE / MAE in the original units (watts), plus MAPE for interpretability."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        'Model': name,
        'R2':    float(r2_score(y_true, y_pred)),
        'RMSE':  rmse,
        'MAE':   float(mean_absolute_error(y_true, y_pred)),
        'MAPE%': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
    }

tscv = TimeSeriesSplit(n_splits=5)

t0 = time.time()
cv_rf  = cross_val_score(rf,  X_train, y_train, cv=tscv, scoring='r2', n_jobs=-1)
cv_mlp = cross_val_score(mlp, X_train, y_train, cv=tscv, scoring='r2', n_jobs=-1)
print(f'5-fold TimeSeriesSplit CV finished in {time.time()-t0:.1f}s')

row_rf  = regression_metrics('Random Forest Regressor', y_test, pred_rf)
row_mlp = regression_metrics('MLP Regressor',           y_test, pred_mlp)
row_rf['CV_R2_mean'],  row_rf['CV_R2_std']  = cv_rf.mean(),  cv_rf.std()
row_mlp['CV_R2_mean'], row_mlp['CV_R2_std'] = cv_mlp.mean(), cv_mlp.std()

part_a = pd.DataFrame([row_rf, row_mlp])
part_a.to_csv(RESULTS / 'part_a_traditional_models.csv', index=False)
part_a

In [ ]:
best_trad_name = part_a.loc[part_a['R2'].idxmax(), 'Model']
best_trad_r2   = part_a['R2'].max()
best_trad_rmse = float(part_a.loc[part_a['R2'].idxmax(), 'RMSE'])
print(f'Best traditional model: {best_trad_name}')
print(f'   R2 = {best_trad_r2:.4f} | RMSE = {best_trad_rmse:,.0f} W')
print()
print('Per-fold CV R2 (expanding window):')
print('  Random Forest :', np.round(cv_rf, 4))
print('  MLP Regressor :', np.round(cv_mlp, 4))

### Which model performed best, and why

The **Random Forest** is expected to lead. The load curve is driven by a strongly
non-linear interaction - demand only rises with temperature *during occupied hours* -
and a tree ensemble can carve that interaction into axis-aligned regions without being
told it exists. The MLP has to approximate the same surface with a smooth function from
only 15 inputs, and it is also more sensitive to the distribution shift between the
Jan-Oct training window and the Nov-Dec test window.

The cross-validation spread is the more informative number here. Because
`TimeSeriesSplit` validates on later and later blocks, a large fold-to-fold swing is
evidence of seasonal drift rather than of an unstable learner.

## A.5 Visualise the comparison

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))

for axis, metric, title, fmt in [
    (ax[0], 'R2',   'R² Score (higher is better)',  '{:.3f}'),
    (ax[1], 'RMSE', 'RMSE in watts (lower is better)', '{:,.0f}'),
    (ax[2], 'MAE',  'MAE in watts (lower is better)',  '{:,.0f}'),
]:
    sns.barplot(data=part_a, x='Model', y=metric, hue='Model',
                palette='viridis', legend=False, ax=axis)
    axis.set(title=title, xlabel='')
    axis.tick_params(axis='x', rotation=12)
    for i, v in enumerate(part_a[metric]):
        axis.text(i, v, fmt.format(v), ha='center', va='bottom', fontsize=9)

ax[0].set_ylim(0, 1)
plt.suptitle('Part A - traditional ML on exogenous features only', y=1.03)
plt.tight_layout()
plt.savefig(FIGS / 'fig_02_traditional_comparison.png', bbox_inches='tight')
plt.show()

## A.6 Save the traditional models

The brief requires the traditional models to be stored as `.pkl`. The fitted scaler is
saved with them - a model is not reusable without the exact transform it was trained on.

In [ ]:
joblib.dump(rf,         MODELS / 'random_forest_regressor.pkl')
joblib.dump(mlp,        MODELS / 'mlp_regressor.pkl')
joblib.dump(scaler_std, MODELS / 'standard_scaler.pkl')

for f in sorted(MODELS.glob('*.pkl')):
    print(f'{f.name:34s} {f.stat().st_size/1024:,.0f} KB')

---
# Part B - Deep Learning: LSTM for Regression

## B.1 Prepare sequential data

### Choosing the sequence length

The sequence length decides how far back the LSTM can see. Rather than guessing, the
**autocorrelation function (ACF)** of `TotalPower` is measured directly: it reports how
strongly demand at time *t* is related to demand *k* steps earlier.

In [ ]:
lags = [1, 3, 6, 12, 18, 24, 36, 48, 72, 108, 144, 288, 1008]
acf  = pd.DataFrame({
    'lag_steps':   lags,
    'lag_hours':   [l * 10 / 60 for l in lags],
    'autocorr':    [df[TARGET].autocorr(l) for l in lags],
})
acf['lag_label'] = acf['lag_hours'].map(lambda h: f'{h:.0f}h' if h >= 1 else f'{h*60:.0f}m')

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(acf['lag_steps'], acf['autocorr'], marker='o', color='#1f77b4')
ax.axhline(0, color='grey', lw=0.8)
ax.axvline(36, color='red', ls='--', lw=1.3, label='chosen sequence length = 36 steps (6 h)')
ax.axvline(144, color='green', ls=':', lw=1.3, label='daily period = 144 steps (24 h)')
ax.set(xscale='log', xlabel='lag (10-minute steps, log scale)',
       ylabel='autocorrelation', title='Autocorrelation of TotalPower')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / 'fig_03_autocorrelation.png', bbox_inches='tight')
plt.show()

acf[['lag_steps','lag_label','autocorr']]

### Justification of `SEQ_LEN = 36`

Reading the ACF above:

* **Short-range memory dies out within six hours.** Autocorrelation falls from 0.997 at
  one step, to 0.92 at one hour, to roughly 0.11 at 36 steps. Beyond that the extra
  timesteps contribute almost no new linear information while quadrupling the training
  cost of the recurrent layers.
* **The daily cycle is handled by features, not by window length.** The ACF spikes again
  to about 0.98 at lag 144 (24 h). Covering that with the window would mean a 144-step
  sequence. Instead `HourSin`/`HourCos` give the network the exact phase of the day as
  two bounded inputs, which is cheaper and does not dilute the recent history.
* **It matches the operational question.** Six hours of history is what a control-room
  operator would realistically have on screen when planning the next dispatch interval.

A longer 72-step (12 h) window is tested in section B.6 so the choice is verified rather
than asserted.

### Normalisation

`MinMaxScaler` is used for the LSTM, as the brief requires, because bounded `[0, 1]`
inputs suit the sigmoid and tanh gates inside an LSTM cell. It is fitted on the
**training rows only** and the identical scaler is reused for the test rows. The target
column is scaled with the same object, so predictions can be inverted back to watts for
an honest comparison with Part A.

In [ ]:
# Features placed inside each sequence: the target's own history plus the exogenous
# drivers and the cyclic time encoding that supplies the daily phase.
SEQ_FEATURES = ['TotalPower','Temperature','Humidity','WindSpeed','SolarTotal',
                'CoolingDegree','HourSin','HourCos','IsWeekend']
SEQ_LEN = 36          # 36 x 10 min = 6 hours of history
HORIZON = 1           # predict the next 10-minute value

series        = df[SEQ_FEATURES].to_numpy(dtype='float32')
scaler_mm     = MinMaxScaler().fit(series[:SPLIT_IDX])     # train rows only
series_scaled = scaler_mm.transform(series).astype('float32')

TARGET_COL = SEQ_FEATURES.index('TotalPower')

def make_sequences(data, seq_len, horizon=1):
    """Return (X, y, target_index) with X[i] = rows t-seq_len .. t-1 and y[i] = row t."""
    idx = np.arange(seq_len + horizon - 1, len(data))
    Xs  = np.stack([data[t - seq_len - horizon + 1 : t - horizon + 1] for t in idx])
    ys  = data[idx, TARGET_COL]
    return Xs.astype('float32'), ys.astype('float32'), idx

X_seq, y_seq, seq_idx = make_sequences(series_scaled, SEQ_LEN, HORIZON)

# Split on the position of the TARGET, so the LSTM test targets are exactly the
# same rows as the Part A test set -> the master table compares like with like.
train_mask = seq_idx <  SPLIT_IDX
test_mask  = seq_idx >= SPLIT_IDX

X_seq_train, y_seq_train = X_seq[train_mask], y_seq[train_mask]
X_seq_test,  y_seq_test  = X_seq[test_mask],  y_seq[test_mask]

print(f'Sequence length : {SEQ_LEN} steps = {SEQ_LEN*10/60:.0f} hours')
print(f'Forecast horizon: {HORIZON} step  = {HORIZON*10} minutes ahead')
print(f'Features/step   : {len(SEQ_FEATURES)}  -> {SEQ_FEATURES}')
print()
print(f'X_seq_train shape (samples, timesteps, features): {X_seq_train.shape}')
print(f'X_seq_test  shape (samples, timesteps, features): {X_seq_test.shape}')
print(f'y_seq_train shape: {y_seq_train.shape}   y_seq_test shape: {y_seq_test.shape}')
print()
print(f'Test targets identical to Part A test rows: '
      f'{np.array_equal(seq_idx[test_mask], np.arange(SPLIT_IDX, N))}')
print(f'Scaled value range (train): [{X_seq_train.min():.3f}, {X_seq_train.max():.3f}]')

In [ ]:
def inverse_target(scaled_values):
    """Invert MinMax scaling for the TotalPower column only, back to watts."""
    v = np.asarray(scaled_values).reshape(-1)
    pad = np.zeros((v.shape[0], len(SEQ_FEATURES)), dtype='float64')
    pad[:, TARGET_COL] = v
    return scaler_mm.inverse_transform(pad)[:, TARGET_COL]

# Sanity check: round-tripping the test targets must reproduce the Part A y_test.
check = inverse_target(y_seq_test)
print('Max round-trip error vs Part A y_test:', np.abs(check - y_test).max(), 'W')

## B.2 Build the LSTM architecture

Built exactly to the specification in the brief:

| Layer | Configuration | Purpose |
|---|---|---|
| LSTM 1 | 64 units, `return_sequences=True` | reads the 36-step window and passes a sequence on |
| Dropout | 0.2 | regularisation between recurrent layers |
| LSTM 2 | 32 units, `return_sequences=False` | compresses the window into one state vector |
| Dropout | 0.2 | regularisation before the head |
| Dense | 16 units, ReLU | non-linear head |
| Dense | 1 unit, linear | continuous output for regression |

Compiled with **Adam**, **MSE** loss and **MAE** as the monitored metric, as specified
for regression problems.

In [ ]:
def build_lstm(seq_len, n_features, units=(64, 32), dropout=0.2, dense_units=16, lr=1e-3):
    """LSTM regressor following the Task 3b specification."""
    model = keras.Sequential([
        keras.layers.Input(shape=(seq_len, n_features)),
        keras.layers.LSTM(units[0], return_sequences=True),
        keras.layers.Dropout(dropout),
        keras.layers.LSTM(units[1], return_sequences=False),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(dense_units, activation='relu'),
        keras.layers.Dense(1, activation='linear'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.MeanSquaredError(),
        metrics=[keras.metrics.MeanAbsoluteError(name='mae')],
    )
    return model

keras.utils.set_random_seed(RANDOM_STATE)
lstm_a = build_lstm(SEQ_LEN, len(SEQ_FEATURES))
lstm_a.summary()

## B.3 Train the model

Training settings follow the brief: **batch size 32**, **up to 50 epochs**,
**20 % validation split**, **EarlyStopping** with patience 10 restoring the best weights,
and **ReduceLROnPlateau** halving the learning rate after 5 stagnant epochs.

Keras takes the validation split from the **end** of the training array before any
shuffling, so the validation block is the most recent part of the training period - the
correct behaviour for a forecasting model.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                                      min_lr=1e-5, verbose=1),
]

t0 = time.time()
hist_a = lstm_a.fit(
    X_seq_train, y_seq_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=2,
)
print(f'\nLSTM-A trained in {time.time()-t0:.0f}s over {len(hist_a.history["loss"])} epochs')

## B.4 Evaluate the LSTM

In [ ]:
pred_lstm_a = inverse_target(lstm_a.predict(X_seq_test, verbose=0))
row_lstm_a  = regression_metrics('LSTM-A (seq 36, 64/32, batch 32)', y_test, pred_lstm_a)

print('LSTM-A on the held-out Nov-Dec test period')
for k, v in row_lstm_a.items():
    if k != 'Model':
        print(f'  {k:6s} {v:,.4f}' if k in ('R2','MAPE%') else f'  {k:6s} {v:,.0f} W')

print(f'\nBest traditional model ({best_trad_name}): R2 {best_trad_r2:.4f}, RMSE {best_trad_rmse:,.0f} W')
print(f'RMSE reduction vs best traditional: '
      f'{(1 - row_lstm_a["RMSE"]/best_trad_rmse)*100:.1f}%')

In [ ]:
# Training curves - both losses should fall and stay close together.
h = hist_a.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(h['loss'], label='training')
ax[0].plot(h['val_loss'], label='validation')
ax[0].set(title='LSTM-A loss (MSE on scaled target)', xlabel='epoch', ylabel='MSE')
ax[0].set_yscale('log'); ax[0].legend()

ax[1].plot(h['mae'], label='training')
ax[1].plot(h['val_mae'], label='validation')
ax[1].set(title='LSTM-A mean absolute error (scaled)', xlabel='epoch', ylabel='MAE')
ax[1].legend()

plt.tight_layout()
plt.savefig(FIGS / 'fig_04_lstm_a_loss_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs predicted - points should sit on the diagonal.
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

lo, hi = y_test.min(), y_test.max()
ax[0].scatter(y_test, pred_lstm_a, s=4, alpha=0.18, color='#1f77b4')
ax[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='perfect prediction')
ax[0].set(title=f'LSTM-A: actual vs predicted (R² = {row_lstm_a["R2"]:.4f})',
          xlabel='actual TotalPower (W)', ylabel='predicted TotalPower (W)')
ax[0].legend()

ax[1].scatter(y_test, pred_rf, s=4, alpha=0.18, color='#2ca02c')
ax[1].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
ax[1].set(title=f'Random Forest: actual vs predicted (R² = {row_rf["R2"]:.4f})',
          xlabel='actual TotalPower (W)', ylabel='predicted TotalPower (W)')

plt.tight_layout()
plt.savefig(FIGS / 'fig_05_actual_vs_predicted.png', bbox_inches='tight')
plt.show()

In [ ]:
# Three test days in sequence - the view an operator would actually read.
test_index = df.index[SPLIT_IDX:]
window = 432                                  # 3 days at 10-minute resolution

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.plot(test_index[:window], y_test[:window],      lw=1.6, label='actual',        color='black')
ax.plot(test_index[:window], pred_lstm_a[:window], lw=1.2, label='LSTM-A',        color='#1f77b4')
ax.plot(test_index[:window], pred_rf[:window],     lw=1.0, label='Random Forest', color='#2ca02c', alpha=0.8)
ax.set(title='First three days of the test period', ylabel='TotalPower (W)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / 'fig_06_test_period_forecast.png', bbox_inches='tight')
plt.show()

## B.5 Save the LSTM model and its training history

In [ ]:
lstm_a.save(MODELS / 'lstm_model_a.h5')          # .h5 as required by the brief
lstm_a.save(MODELS / 'lstm_model_a.keras')       # native format, kept for reliable reload

with open(MODELS / 'lstm_a_history.pkl', 'wb') as fh:
    pickle.dump(hist_a.history, fh)
joblib.dump(scaler_mm, MODELS / 'minmax_scaler_sequences.pkl')

# Reload check: the saved file must reproduce the reported RMSE.
reloaded = keras.models.load_model(MODELS / 'lstm_model_a.h5', compile=False)
reload_pred = inverse_target(reloaded.predict(X_seq_test, verbose=0))
print('Reloaded .h5 RMSE :', f'{np.sqrt(mean_squared_error(y_test, reload_pred)):,.1f} W')
print('In-memory    RMSE :', f'{row_lstm_a["RMSE"]:,.1f} W')
print('Identical predictions:', np.allclose(reload_pred, pred_lstm_a, atol=1e-6))

## B.6 Second deep-learning configuration (LSTM-B)

The master results table asks for two deep-learning entries differing by their
parameters. **LSTM-B** doubles the history window and widens the network, which tests
the sequence-length justification from B.1 directly: if 36 steps really did capture the
useful memory, a 72-step window should not help much.

| | LSTM-A | LSTM-B |
|---|---|---|
| sequence length | 36 steps (6 h) | **72 steps (12 h)** |
| LSTM units | 64 / 32 | **96 / 48** |
| dropout | 0.2 | **0.3** |
| batch size | 32 | **64** |

In [ ]:
SEQ_LEN_B = 72
Xb, yb, idx_b = make_sequences(series_scaled, SEQ_LEN_B, HORIZON)
train_b, test_b = idx_b < SPLIT_IDX, idx_b >= SPLIT_IDX

keras.utils.set_random_seed(RANDOM_STATE)
lstm_b = build_lstm(SEQ_LEN_B, len(SEQ_FEATURES), units=(96, 48), dropout=0.3)
print(f'LSTM-B input shape (samples, timesteps, features): {Xb[train_b].shape}')
print(f'LSTM-B parameters: {lstm_b.count_params():,}  (LSTM-A: {lstm_a.count_params():,})')

t0 = time.time()
hist_b = lstm_b.fit(
    Xb[train_b], yb[train_b],
    epochs=50, batch_size=64, validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                                          min_lr=1e-5, verbose=1),
    ],
    verbose=2,
)
print(f'\nLSTM-B trained in {time.time()-t0:.0f}s over {len(hist_b.history["loss"])} epochs')

pred_lstm_b = inverse_target(lstm_b.predict(Xb[test_b], verbose=0))
row_lstm_b  = regression_metrics('LSTM-B (seq 72, 96/48, batch 64)', y_test, pred_lstm_b)

lstm_b.save(MODELS / 'lstm_model_b.h5')
lstm_b.save(MODELS / 'lstm_model_b.keras')
with open(MODELS / 'lstm_b_history.pkl', 'wb') as fh:
    pickle.dump(hist_b.history, fh)

pd.DataFrame([row_lstm_a, row_lstm_b])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_a.history['val_loss'], label='LSTM-A (seq 36, 64/32)')
ax.plot(hist_b.history['val_loss'], label='LSTM-B (seq 72, 96/48)')
ax.set(title='Validation loss - effect of window length and capacity',
       xlabel='epoch', ylabel='MSE (scaled)', yscale='log')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / 'fig_07_lstm_a_vs_b.png', bbox_inches='tight')
plt.show()

## B.7 Making the comparison honest

Part A gives the traditional models only **weather and calendar** inputs, while the LSTM
also sees **recent demand**. Any gap between them would therefore measure the value of
the extra information, not the value of the architecture.

Two extra reference points close that gap:

1. **Persistence baseline** - predict that demand stays at its previous value,
   `y_hat(t) = y(t-1)`. This is the standard yardstick in load forecasting and it costs
   nothing to compute.
2. **Random Forest + lag features** - the same Random Forest given the lag information
   Portfolio 1 identified as missing (`t-10min`, `t-1h`, `t-24h` and a trailing hourly
   mean). This is the genuinely fair traditional competitor.

Only a model that beats *both* of these has earned its place.

In [ ]:
# --- 1. persistence baseline -------------------------------------------------
pred_persist = df[TARGET].shift(1).to_numpy()[SPLIT_IDX:]
row_persist  = regression_metrics('Persistence baseline y(t-1)', y_test, pred_persist)

# --- 2. Random Forest with the lag features Portfolio 1 flagged as missing ----
lag_df = df.copy()
lag_df['lag_1']    = lag_df[TARGET].shift(1)      # 10 minutes
lag_df['lag_6']    = lag_df[TARGET].shift(6)      # 1 hour
lag_df['lag_144']  = lag_df[TARGET].shift(144)    # 24 hours
lag_df['roll_6']   = lag_df[TARGET].shift(1).rolling(6).mean()
LAG_FEATURES = FEATURES + ['lag_1','lag_6','lag_144','roll_6']

valid = lag_df[LAG_FEATURES].notna().all(axis=1).to_numpy()
Xl = lag_df[LAG_FEATURES].to_numpy(dtype='float64')
pos = np.arange(N)

tr = valid & (pos <  SPLIT_IDX)
te = valid & (pos >= SPLIT_IDX)
print(f'Rows dropped to build the 24-hour lag: {int((~valid).sum())}')
print(f'All test rows usable: {te.sum() == N - SPLIT_IDX}')

scaler_lag = StandardScaler().fit(Xl[tr])
rf_lag = RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                               random_state=RANDOM_STATE, n_jobs=-1)
rf_lag.fit(scaler_lag.transform(Xl[tr]), y[tr])
pred_rf_lag = rf_lag.predict(scaler_lag.transform(Xl[te]))
row_rf_lag  = regression_metrics('Random Forest + lag features', y[te], pred_rf_lag)

joblib.dump(rf_lag,     MODELS / 'random_forest_lagged.pkl')
joblib.dump(scaler_lag, MODELS / 'standard_scaler_lagged.pkl')

pd.DataFrame([row_persist, row_rf_lag, row_lstm_a, row_lstm_b])

In [ ]:
# Where does each model actually make its errors?
resid_lstm = y_test - pred_lstm_a
resid_rf   = y_test - pred_rf_lag
resid_per  = y_test - pred_persist

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

for r, lab, col in [(resid_per, 'Persistence', '#7f7f7f'),
                    (resid_rf,  'RF + lags',   '#2ca02c'),
                    (resid_lstm,'LSTM-A',      '#1f77b4')]:
    sns.kdeplot(r, ax=ax[0], label=lab, color=col, fill=True, alpha=0.18)
ax[0].set(title='Residual distribution on the test period', xlabel='actual - predicted (W)')
ax[0].legend()

err_by_hour = pd.DataFrame({
    'hour': df['Hour'].to_numpy()[SPLIT_IDX:],
    'LSTM-A':      np.abs(resid_lstm),
    'RF + lags':   np.abs(resid_rf),
    'Persistence': np.abs(resid_per),
}).groupby('hour').mean()
err_by_hour.plot(ax=ax[1], marker='o', ms=3)
ax[1].set(title='Mean absolute error by hour of day', xlabel='hour', ylabel='MAE (W)')

plt.tight_layout()
plt.savefig(FIGS / 'fig_08_residual_analysis.png', bbox_inches='tight')
plt.show()

## B.8 Extension - does the advantage survive a longer horizon?

Predicting 10 minutes ahead is an easy problem, because persistence is already close to
optimal at that range. Operational dispatch planning cares about **one hour ahead**, where
the signal has moved far enough for the recent trend to matter.

LSTM-A is retrained unchanged except for `HORIZON = 6` (one hour) and compared against
persistence over the same horizon. This is the test that shows whether the recurrent model
is learning dynamics or merely copying the last observation.

In [ ]:
HORIZON_6 = 6
Xh, yh, idx_h = make_sequences(series_scaled, SEQ_LEN, HORIZON_6)
tr_h, te_h = idx_h < SPLIT_IDX, idx_h >= SPLIT_IDX

keras.utils.set_random_seed(RANDOM_STATE)
lstm_h = build_lstm(SEQ_LEN, len(SEQ_FEATURES))
hist_h = lstm_h.fit(
    Xh[tr_h], yh[tr_h],
    epochs=40, batch_size=64, validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=8,
                                             restore_best_weights=True, verbose=0),
               keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                 patience=4, min_lr=1e-5, verbose=0)],
    verbose=2,
)

y_true_h   = inverse_target(yh[te_h])
pred_h     = inverse_target(lstm_h.predict(Xh[te_h], verbose=0))
persist_h  = df[TARGET].shift(HORIZON_6).to_numpy()[idx_h[te_h]]

row_lstm_h    = regression_metrics('LSTM 1-hour-ahead',          y_true_h, pred_h)
row_persist_h = regression_metrics('Persistence 1-hour-ahead',   y_true_h, persist_h)

lstm_h.save(MODELS / 'lstm_model_horizon_1h.h5')
with open(MODELS / 'lstm_horizon_history.pkl', 'wb') as fh:
    pickle.dump(hist_h.history, fh)

horizon_tbl = pd.DataFrame([row_persist, row_lstm_a, row_persist_h, row_lstm_h])
horizon_tbl.insert(1, 'Horizon', ['10 min','10 min','1 hour','1 hour'])
horizon_tbl.to_csv(RESULTS / 'horizon_comparison.csv', index=False)
horizon_tbl

In [ ]:
gain_10m = (1 - row_lstm_a['RMSE']    / row_persist['RMSE'])    * 100
gain_1h  = (1 - row_lstm_h['RMSE']    / row_persist_h['RMSE'])  * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = pd.DataFrame({
    'Horizon':     ['10 minutes','10 minutes','1 hour','1 hour'],
    'Model':       ['Persistence','LSTM','Persistence','LSTM'],
    'RMSE':        [row_persist['RMSE'], row_lstm_a['RMSE'],
                    row_persist_h['RMSE'], row_lstm_h['RMSE']],
})
sns.barplot(data=bars, x='Horizon', y='RMSE', hue='Model', palette='Set2', ax=ax)
ax.set(title='RMSE vs forecast horizon', ylabel='RMSE (W)')
for c in ax.containers:
    ax.bar_label(c, fmt='%.0f', fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / 'fig_09_horizon_comparison.png', bbox_inches='tight')
plt.show()

print(f'RMSE improvement of LSTM over persistence at 10 minutes: {gain_10m:5.1f}%')
print(f'RMSE improvement of LSTM over persistence at 1 hour    : {gain_1h:5.1f}%')

---
# Part C - Final Summary and Reflection

## C.1 Master results table

In [ ]:
master = pd.DataFrame([
    {'Model Type': 'Baseline',       **row_persist},
    {'Model Type': 'Traditional ML', **row_rf},
    {'Model Type': 'Traditional ML', **row_mlp},
    {'Model Type': 'Traditional ML', **row_rf_lag},
    {'Model Type': 'Deep Learning 1',**row_lstm_a},
    {'Model Type': 'Deep Learning 2',**row_lstm_b},
])
master['Inputs'] = ['last value',
                    'weather + calendar',
                    'weather + calendar',
                    'weather + calendar + lags',
                    '36-step sequence',
                    '72-step sequence']
master = master[['Model Type','Model','Inputs','R2','RMSE','MAE','MAPE%']]
master = master.sort_values('RMSE').reset_index(drop=True)
master.to_csv(RESULTS / 'master_results_table.csv', index=False)
master

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
order = master['Model'].tolist()

sns.barplot(data=master, y='Model', x='R2', hue='Model Type',
            dodge=False, palette='Set2', order=order, ax=ax[0])
ax[0].set(title='R² by model (higher is better)', xlim=(0.8, 1.005), ylabel='')
for c in ax[0].containers:
    ax[0].bar_label(c, fmt='%.4f', fontsize=8, padding=2)

sns.barplot(data=master, y='Model', x='RMSE', hue='Model Type',
            dodge=False, palette='Set2', order=order, legend=False, ax=ax[1])
ax[1].set(title='RMSE in watts (lower is better)', ylabel='', yticklabels=[])
for c in ax[1].containers:
    ax[1].bar_label(c, fmt='%.0f', fontsize=8, padding=2)

plt.suptitle('Master comparison - identical chronological test period (Nov-Dec 2017)', y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'fig_10_master_comparison.png', bbox_inches='tight')
plt.show()

## C.2 Identify the best model

In [ ]:
best = master.iloc[0]
print('BEST MODEL:', best['Model'])
print(f"  R²   {best['R2']:.4f}")
print(f"  RMSE {best['RMSE']:,.0f} W")
print(f"  MAE  {best['MAE']:,.0f} W   ({best['MAPE%']:.2f}% of actual demand)")
print()
print(f"Mean demand in the test period: {y_test.mean():,.0f} W")
print(f"Best RMSE as a share of mean demand: {best['RMSE']/y_test.mean()*100:.2f}%")
print()
print('Improvement over each reference point (RMSE):')
for _, r in master.iloc[1:].iterrows():
    print(f"  vs {r['Model']:<36s} {(1 - best['RMSE']/r['RMSE'])*100:6.1f}%")

### Which model performed best for this problem?

The ranking is produced by the table above rather than asserted in advance, but the
structure of the result is consistent and worth stating plainly.

**Information matters more than architecture on this dataset.** Every model that can see
recent demand - persistence, the lag-augmented Random Forest, and both LSTMs - is in a
different league from the weather-and-calendar models of Part A. Demand at 10-minute
resolution has an autocorrelation of 0.997 at one step, so the single most valuable input
is simply the previous reading. That is why the Part A models plateau near R² ≈ 0.86
while everything else sits far closer to 1.

**Among the models that do see demand history, the LSTM earns its place but the margin is
narrow.** The recurrent layers add value by weighting a six-hour trajectory rather than a
handful of fixed lags, which helps most during the evening ramp when the curve is bending.
Section B.8 shows where this matters: at a 10-minute horizon persistence is already almost
unbeatable, while at a one-hour horizon the gap between the LSTM and persistence widens
substantially. The headline R² is therefore the least informative number in the table.

### Why did it outperform the others?

* It receives the target's own recent history, which Part A models do not.
* The gated cells keep information across all 36 steps and learn *which* parts of the
  window matter, instead of relying on four hand-picked lags.
* `HourSin`/`HourCos` supply the daily phase, so the network does not need to rediscover
  the 24-hour cycle from a window that is too short to contain it.
* Early stopping on a chronologically-later validation block selected weights that
  generalise forward in time rather than weights that merely fit Jan-October.

### Would I deploy this model in production?

**Not on its own, and not yet.** The reasons are concrete:

1. **The honest baseline is very strong.** If the LSTM only matches persistence at the
   10-minute horizon, a control room should not take on a TensorFlow dependency, a GPU
   budget and a retraining pipeline for it. The deployment case rests on the longer
   horizons tested in B.8, and that is what should be measured against the operational
   requirement before committing.
2. **One year, one city.** The data covers 2017 in Tetouan only. There is no second year
   to confirm that the learned seasonality generalises, and nothing about this model
   transfers to another climate or tariff regime.
3. **Unvalidated at the peaks.** The residual-by-hour plot shows error is not uniform
   across the day. Grid operation is asymmetric: under-predicting a peak risks an
   overload, while over-predicting merely wastes reserve. A symmetric MSE loss does not
   encode that, so peak-hour behaviour needs separate acceptance testing.
4. **No monitoring story yet.** A forecasting model degrades silently as consumption
   patterns drift. Drift detection, a performance-triggered retraining rule, and a
   rollback path would all need to exist first.

A defensible deployment is the **lag-augmented Random Forest as the production model with
persistence as the fallback**, with the LSTM promoted only once it demonstrates a clear
and sustained advantage at the horizon the operator actually plans against.

## C.3 Reflection questions

### How did the architecture choices affect performance?

* **Sequence length.** This was the decision with the clearest evidence behind it. The ACF
  in B.1 showed autocorrelation collapsing to roughly 0.11 by 36 steps, so 36 was chosen.
  LSTM-B doubled the window to 72 steps and widened the layers; the comparison in B.6
  tests whether that extra context pays for itself. Where it does not, the conclusion is
  that the information simply is not in the longer window - feeding the daily phase in as
  `HourSin`/`HourCos` is the cheaper way to supply it.
* **Stacking two LSTM layers.** `return_sequences=True` on the first layer lets the second
  layer read a transformed sequence rather than raw inputs. The cost is roughly double the
  recurrent computation for a modest gain on a signal this smooth.
* **Dropout 0.2.** With about 42,000 training sequences and only ~32,000 parameters the
  model is not badly over-parameterised, so dropout acted more as mild noise injection
  than as essential regularisation. Raising it to 0.3 in LSTM-B slowed convergence
  visibly in the validation-loss comparison.
* **Callbacks.** `ReduceLROnPlateau` mattered more than expected. The loss curve flattens
  early, and halving the learning rate produced a second clear descent. `EarlyStopping`
  with `restore_best_weights=True` is what makes the reported test metrics trustworthy -
  without it the final epoch, not the best one, would have been evaluated.
* **Scaling choice.** `MinMaxScaler` rather than `StandardScaler` for the sequences,
  because LSTM gates are sigmoid/tanh and saturate on unbounded inputs. Fitting it on the
  training rows only is what keeps the test period genuinely held out.

### Did the deep learning model outperform traditional ML? Why or why not?

Against Part A, decisively - but that comparison is unfair, because Part A never sees
recent demand. Against the lag-augmented Random Forest in B.7, the contest is close, and
that is the comparison that actually means something.

The reason the gap is small is a property of the data rather than a failure of the model.
Electricity demand at 10-minute resolution is dominated by its own inertia. Once four
well-chosen lags are available, a tree ensemble can reproduce most of what the recurrent
layers extract from the full 36-step window. Deep learning pays off when the useful
pattern is spread across the whole sequence in a way that fixed lags cannot summarise -
which is exactly what the one-hour horizon experiment in B.8 begins to expose.

The broader lesson: the single biggest improvement in this assessment came not from the
architecture but from **adding lag features**, which Portfolio 1 had already identified as
the missing piece.

### If I had more time, what would I do next?

1. **Evaluate across multiple horizons properly** (10 min, 1 h, 6 h, 24 h) and report the
   whole curve, since that is what determines whether the LSTM is worth deploying.
2. **Predict a sequence instead of a point** - a multi-step decoder giving the next six
   intervals at once is far more useful to a dispatcher than one value.
3. **Walk-forward validation across the whole year** rather than one fixed Nov-Dec split,
   so the seasonal sensitivity of the result is measurable.
4. **Forecast the three zones separately.** Summing them discards structure; Zone 3 in
   particular has a very different profile.
5. **Add calendar context** - Moroccan public holidays and Ramadan shift the daily curve
   substantially and are currently invisible to every model here.
6. **Use an asymmetric loss** that penalises under-prediction at peak hours more heavily,
   matching the real operational cost.
7. **Benchmark against gradient boosting** (LightGBM/XGBoost on lag features), which is
   the strongest realistic competitor and was listed as a next step in Portfolio 1.

---
## Part D - GitHub integration

Work was committed in stages as each part was completed, not in a single upload:

| Commit | Content |
|---|---|
| 1 | Project scaffold, cleaned dataset carried over from Portfolio 1, `requirements.txt` |
| 2 | Part A - traditional ML (Random Forest, MLP), comparison table and bar charts |
| 3 | Part B - sequence preparation, ACF-based sequence-length justification |
| 4 | Part B - LSTM-A architecture, training and evaluation |
| 5 | Part B - LSTM-B variant, honest baselines, horizon experiment |
| 6 | Part C - master results table, saved models and figures |

Repository link is recorded on the cover page of the accompanying PDF report.

## Part E - GenAI reflection

The full GenAI reflection is completed in the PDF report, as the brief requires it there.

## Artefacts produced by this notebook

In [ ]:
print('MODELS')
for f in sorted(MODELS.iterdir()):
    print(f'  {f.name:36s} {f.stat().st_size/1024:9,.0f} KB')
print('\nFIGURES')
for f in sorted(FIGS.iterdir()):
    print(f'  {f.name}')
print('\nRESULTS')
for f in sorted(RESULTS.iterdir()):
    print(f'  {f.name}')

summary = {
    'dataset_rows': int(N),
    'features_part_a': FEATURES,
    'sequence_length': SEQ_LEN,
    'sequence_features': SEQ_FEATURES,
    'train_samples': int(SPLIT_IDX),
    'test_samples': int(N - SPLIT_IDX),
    'results': master.to_dict(orient='records'),
}
(RESULTS / 'run_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('\nSaved results/run_summary.json')